# ResNet Deep Learning on DICOM Images
This notebook demonstrates training a **ResNet-18** model on DICOM medical images using PyTorch.
It covers: loading DICOM files → preprocessing → building a ResNet model → training → evaluation.

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required packages
!pip install torch torchvision pydicom pillow numpy matplotlib scikit-learn tqdm --quiet

In [ ]:
import os
import numpy as np
import pydicom
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. DICOM Dataset Loader
The dataset expects DICOM files organized in class subdirectories:
```
ROOT_DIR/
    class_A/   ← e.g. CT
        file1.dcm
    class_B/   ← e.g. MR
        file2.dcm
```

In [ ]:
class DicomDataset(Dataset):
    """PyTorch Dataset for DICOM images organized in class subdirectories."""

    DICOM_EXTS = ('.dcm', '.dcn', '.kon', '.pr', '.sr')

    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []   # list of (file_path, label_idx)
        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for cls in self.classes:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith(self.DICOM_EXTS):
                    self.samples.append(
                        (os.path.join(cls_dir, fname), self.class_to_idx[cls])
                    )

        print(f'Found {len(self.samples)} files across {len(self.classes)} classes: {self.classes}')

    def _read_dicom_as_rgb(self, path):
        ds = pydicom.dcmread(path, force=True)
        arr = ds.pixel_array.astype(np.float32)

        # Normalize to 0-255
        arr_min, arr_max = arr.min(), arr.max()
        if arr_max > arr_min:
            arr = (arr - arr_min) / (arr_max - arr_min) * 255.0
        arr = arr.astype(np.uint8)

        # Convert grayscale to RGB for ResNet
        img = Image.fromarray(arr).convert('RGB')
        return img

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = self._read_dicom_as_rgb(path)
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
# ImageNet normalization stats (ResNet was pretrained on ImageNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('Transforms defined.')

In [ ]:
# ── Set your DICOM root directory ──────────────────────────────────────────
ROOT_DIR = input('Enter path to DICOM root directory (with class subfolders): ').strip().strip('"').strip("'")

full_dataset = DicomDataset(ROOT_DIR, transform=train_transform)

# 80/20 train-validation split
n_val   = max(1, int(0.2 * len(full_dataset)))
n_train = len(full_dataset) - n_val
train_ds, val_ds = random_split(full_dataset, [n_train, n_val])

# Override transform for validation set
val_ds.dataset.transform = val_transform

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

NUM_CLASSES = len(full_dataset.classes)
print(f'Train: {n_train} | Val: {n_val} | Classes: {NUM_CLASSES}')

## 2. ResNet-18 Model (Transfer Learning)
We load a **pretrained ResNet-18** and replace the final fully-connected layer to match our number of classes.

In [ ]:
def build_resnet(num_classes, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    if freeze_backbone:
        # Freeze all layers except the final FC layer
        for param in model.parameters():
            param.requires_grad = False

    # Replace the classifier head
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes)
    )
    return model

model = build_resnet(NUM_CLASSES, freeze_backbone=True).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

## 3. Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0.0, 0
    for imgs, labels in tqdm(loader, desc='Train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    all_preds, all_labels = [], []
    for imgs, labels in tqdm(loader, desc='Val  ', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader.dataset), correct / len(loader.dataset), all_preds, all_labels

print('Training functions defined.')

In [ ]:
NUM_EPOCHS = 10
LR         = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), 'best_resnet_dicom.pth')

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS}  '
          f'Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}  |  '
          f'Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}')

print(f'\nBest Val Accuracy: {best_val_acc:.4f}  → saved to best_resnet_dicom.pth')

## 4. Training Curves

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss')
axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val Loss')
axes[0].set_title('Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs, history['train_acc'], 'b-o', label='Train Acc')
axes[1].plot(epochs, history['val_acc'],   'r-o', label='Val Acc')
axes[1].set_title('Accuracy per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 5. Final Evaluation & Classification Report

In [ ]:
# Load the best checkpoint
model.load_state_dict(torch.load('best_resnet_dicom.pth', map_location=DEVICE))

_, _, all_preds, all_labels = evaluate(model, val_loader, criterion)

print('\n── Classification Report ──')
print(classification_report(all_labels, all_preds, target_names=full_dataset.classes))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(full_dataset.classes, rotation=45, ha='right')
ax.set_yticklabels(full_dataset.classes)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 6. Single-Image Inference

In [ ]:
def predict_dicom(model, dicom_path, transform, classes, device):
    ds = pydicom.dcmread(dicom_path, force=True)
    arr = ds.pixel_array.astype(np.float32)
    arr = ((arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * 255).astype(np.uint8)
    img = Image.fromarray(arr).convert('RGB')

    tensor = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = probs.argmax()

    # Display
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(arr, cmap='gray')
    axes[0].set_title(f'DICOM: {os.path.basename(dicom_path)}')
    axes[0].axis('off')
    axes[1].barh(classes, probs, color='steelblue')
    axes[1].set_xlabel('Probability')
    axes[1].set_title(f'Prediction: {classes[pred_idx]} ({probs[pred_idx]:.2%})')
    axes[1].set_xlim(0, 1)
    plt.tight_layout()
    plt.show()

    return classes[pred_idx], probs

# ── Run inference on a single file ────────────────────────────────────────
sample_path = input('Enter path to a single DICOM file for inference: ').strip().strip('"').strip("'")
predicted_class, probabilities = predict_dicom(model, sample_path, val_transform, full_dataset.classes, DEVICE)
print(f'\nPredicted: {predicted_class}')
for cls, prob in zip(full_dataset.classes, probabilities):
    print(f'  {cls}: {prob:.4f}')